# 05 — `yield` and Generator Functions

This notebook moves from **“what is a generator?”** to understanding **`yield`** and how generator functions pause, resume, and preserve their state.

> **Progression:** `01_Iterables_and_Iterators` → `02_Iterator_Protocol` → `03_Custom_Iterators` → `04_Generators` → **`05_Yield_and_Generator_Functions`**

## 1. Introduction to `yield`

`yield` is used inside a generator function to produce a value while preserving the function's execution state.

Simple example:

```python
def numbers():
    yield 1
    yield 2
    yield 3
```

Calling:

```python
result = numbers()
```

does **not** immediately execute the function body. It creates a generator object.

The function begins executing when the generator is advanced, such as with `next()` or a `for` loop.

In [ ]:
def numbers():
    yield 1
    yield 2
    yield 3


result = numbers()

print(result)
print(type(result))

## 2. `yield` vs `return`

This is one of the most important distinctions.

### Normal function

```python
def get_number():
    return 10
```

### Generator function

```python
def get_numbers():
    yield 10
    yield 20
    yield 30
```

| `return` | `yield` |
|---|---|
| Returns a result | Produces a value |
| Function ends | Function pauses |
| Cannot resume after returning | Can resume later |
| Usually one final result | Can produce many values |
| Used by a normal function | Makes the function a generator function |

Compare their execution:

In [ ]:
def test_return():
    print("Start")
    return 10
    print("End")


def test_yield():
    print("Start")
    yield 10
    print("End")


print("Calling normal function:")
result = test_return()
print("Result:", result)

print()
print("Calling generator function:")
generator = test_yield()
print("Generator created:", generator)

Notice the difference:

- `test_return()` executes immediately and reaches `return`.
- `test_yield()` creates a generator object without executing its body yet.

Now advance the generator:

In [ ]:
print(next(generator))
print(next(generator))

The first `next()` runs until the first `yield`.

The second `next()` resumes after that `yield`, prints `"End"`, and then the function finishes. Because there is no more `yield`, that second call raises `StopIteration` after printing `"End"`.

## 3. How `yield` Pauses Execution

Consider:

```python
def numbers():
    print("Step 1")
    yield 1

    print("Step 2")
    yield 2

    print("Step 3")
    yield 3
```

Each call to `next()` runs only until the next `yield`.

In [ ]:
def numbers():
    print("Step 1")
    yield 1

    print("Step 2")
    yield 2

    print("Step 3")
    yield 3


generator = numbers()

print(next(generator))
print(next(generator))
print(next(generator))

Expected output:

```text
Step 1
1
Step 2
2
Step 3
3
```

The execution flow is:

```text
next()
   ↓
execute until yield
   ↓
produce value
   ↓
pause
   ↓
next()
   ↓
resume from previous yield
```

## 4. How `yield` Resumes Execution

Execution does **not** restart from the beginning.

```python
def demo():
    print("A")
    yield 1

    print("B")
    yield 2

    print("C")
    yield 3
```

Create the generator and advance it step by step:

In [ ]:
def demo():
    print("A")
    yield 1

    print("B")
    yield 2

    print("C")
    yield 3


g = demo()

next(g)
next(g)
next(g)

The second `next()` resumes **after the first `yield`**, and the third resumes after the second `yield`.

The generator remembers where execution was paused.

## 5. Generator Function vs Normal Function

The presence of `yield` makes a function a **generator function**.

Compare:

```python
def normal_function():
    print("Start")
    return 10
```

with:

```python
def generator_function():
    print("Start")
    yield 10
```

Calling the normal function executes its body immediately. Calling the generator function creates a generator object; its body begins when the generator is advanced.

In [ ]:
def normal_function():
    print("Start")
    return 10


def generator_function():
    print("Start")
    yield 10


print("Normal function call:")
normal_result = normal_function()
print("Returned:", normal_result)

print()
print("Generator function call:")
generator_result = generator_function()
print("Created:", generator_result)

print()
print("Advance generator:")
print(next(generator_result))

The key distinction is:

```text
Normal function
    ↓
executes
    ↓
return
    ↓
finishes

Generator function
    ↓
creates generator object
    ↓
next()
    ↓
executes until yield
    ↓
pauses
```

## 6. Generator Object

A generator function creates a generator object when it is called.

In [ ]:
def numbers():
    yield 1
    yield 2
    yield 3


g = numbers()

print(type(g))
print(iter(g) is g)

The important relationship is:

```text
generator function
        ↓
   generator object
        ↓
      iterator
```

> **A generator object is an iterator.**

That means it follows the iterator behavior introduced in the earlier notebooks and can be consumed with `next()` or a `for` loop.

## 7. Generator State

Generators preserve local variables between `yield` points.

Consider:

```python
def counter():
    count = 1

    while count <= 3:
        yield count
        count += 1
```

The local variable `count` does not disappear when the generator pauses.

In [ ]:
def counter():
    count = 1

    while count <= 3:
        yield count
        count += 1


g = counter()

print(next(g))
print(next(g))
print(next(g))

The generator progresses like this:

```text
count = 1
   ↓
yield 1
   ↓
pause
   ↓
resume
   ↓
count becomes 2
   ↓
yield 2
   ↓
pause
   ↓
resume
   ↓
count becomes 3
   ↓
yield 3
```

This preserved local state is why a generator can resume where it stopped.

## 8. Multiple `yield` Statements

A generator function can contain multiple `yield` statements.

In [ ]:
def values():
    yield "A"
    yield "B"
    yield "C"


g = values()

for value in g:
    print(value)

Expected output:

```text
A
B
C
```

Another example:

In [ ]:
def data():
    yield 10
    yield 20
    yield 30


for value in data():
    print(value)

A generator can therefore produce many values without constructing one large result collection first.

## 9. `yield` Inside Loops

A very common generator pattern is placing `yield` inside a loop.

For example:

In [ ]:
def squares(n):
    for number in range(1, n + 1):
        yield number ** 2


for value in squares(5):
    print(value)

Output:

```text
1
4
9
16
25
```

In [ ]:
def even_numbers(n):
    for number in range(n + 1):
        if number % 2 == 0:
            yield number


for number in even_numbers(10):
    print(number)

Output:

```text
0
2
4
6
8
10
```

The loop and `yield` work together: each matching value is produced as the generator is consumed.

## 10. `yield` with Conditions

Generators can also contain conditions that decide which values should be produced.

In [ ]:
def positive_numbers(numbers):
    for number in numbers:
        if number > 0:
            yield number


values = [-3, 5, -1, 8, 0, 4]

for value in positive_numbers(values):
    print(value)

Output:

```text
5
8
4
```

This is a useful way to build a **lazy filter**: values are produced only when the generator is consumed.

## 11. Sending Values into a Generator

Generators can also receive values through:

```python
generator.send(value)
```

Start with a simple receiver:

```python
def receiver():
    value = yield
    print("Received:", value)
```

The expression:

```python
value = yield
```

does two things:

1. pauses the generator
2. receives a value when `.send()` resumes it

First, the generator must be started so it reaches the `yield`:

In [ ]:
def receiver():
    value = yield
    print("Received:", value)


g = receiver()

next(g)
g.send(100)

The flow is:

```text
next(g)
   ↓
generator reaches yield
   ↓
generator pauses
   ↓
g.send(100)
   ↓
yield expression receives 100
   ↓
value = 100
   ↓
generator continues
```

This is an introductory look at generator communication. Advanced coroutine patterns are outside the scope of this notebook.

## 12. `yield` and Generator Communication

We can build a small stateful calculator using `.send()`.

In [ ]:
def calculator():
    total = 0

    while True:
        value = yield total
        total += value


g = calculator()

print(next(g))
print(g.send(10))
print(g.send(20))
print(g.send(5))

Expected output:

```text
0
10
30
35
```

The communication works in both directions:

```text
generator → yields value
caller    → sends value
generator → resumes
generator → processes value
generator → yields again
```

This demonstrates that `yield` can be more than a one-way way of producing values: a paused generator can also receive a value when it resumes.

## 13. Generator Function with Arguments

Generator functions can accept normal function arguments.

For example:

In [ ]:
def count(start, stop):
    while start <= stop:
        yield start
        start += 1


for number in count(3, 7):
    print(number)

Output:

```text
3
4
5
6
7
```

Another example: generate multiples of a number.

In [ ]:
def multiples(number, limit):
    for value in range(number, limit + 1, number):
        yield value


for value in multiples(5, 25):
    print(value)

Output:

```text
5
10
15
20
25
```

## 14. Practical Generator Functions

Here are several useful patterns.

### Generate squares

In [ ]:
def squares(n):
    for number in range(1, n + 1):
        yield number ** 2


for value in squares(5):
    print(value)

### Generate even numbers

In [ ]:
def evens(n):
    for number in range(0, n + 1, 2):
        yield number


for value in evens(10):
    print(value)

### Generate Fibonacci numbers

In [ ]:
def fibonacci(n):
    a = 0
    b = 1

    for _ in range(n):
        yield a
        a, b = b, a + b


for number in fibonacci(8):
    print(number)

Expected output:

```text
0
1
1
2
3
5
8
13
```

### Generate characters

In [ ]:
def characters(text):
    for character in text:
        yield character


for character in characters("Python"):
    print(character)

Output:

```text
P
y
t
h
o
n
```

## 15. Common Mistakes

### Mistake 1 — Expecting a normal return value

```python
def numbers():
    yield 1
    yield 2
```

This:

```python
print(numbers())
```

prints the generator object, not `1` and `2`.

Consume the generator with `next()` or a `for` loop.

### Mistake 2 — Forgetting that generators are consumed

```python
g = numbers()

print(list(g))
print(list(g))
```

The second result is:

```text
[]
```

because the generator has already been exhausted.

### Mistake 3 — Calling `next()` after exhaustion

```python
g = numbers()

print(next(g))
print(next(g))
print(next(g))
```

The final call raises `StopIteration`.

### Mistake 4 — Confusing `yield` with `return`

```python
def wrong():
    return 1
    return 2
```

Only the first `return` can be reached because `return` ends the function.

To produce multiple values over time, use `yield`.

## 16. Summary

### Key Takeaways

- `yield` turns a function into a generator function.
- Calling a generator function creates a generator object.
- A generator object is an iterator.
- `yield` produces a value and pauses execution.
- `next()` resumes the generator.
- Generator state is preserved between `yield` points.
- A generator can contain multiple `yield` statements.
- `yield` works naturally inside loops and conditions.
- `send()` can send values into a paused generator.
- Generators are consumed as they are iterated.
- An exhausted generator cannot be restarted.

### Mental model

```text
Generator Function
        │
        │ called
        ▼
Generator Object
        │
        │ next()
        ▼
      yield
        │
        ▼
      value
        │
        ▼
     pauses
        │
        │ next()
        ▼
    resumes
        │
        ▼
      yield
        │
        ▼
      ...
        │
        ▼
 StopIteration
```

### Curriculum progression

```text
01 Iterables & Iterators
        ↓
02 Iterator Protocol
        ↓
03 Custom Iterators
        ↓
04 Generators
        ↓
05 yield & Generator Functions   ← current
        ↓
06 Generator Expressions
        ↓
07 Iterators vs Generators
```

### Scope boundary

Generator expressions `(x for x in ...)` belong in **06 Generator Expressions**.

Detailed iterator-vs-generator comparison and use cases belong in **07 Iterators vs Generators**.

Advanced coroutine concepts are not needed here.

`yield from` is intentionally not introduced deeply yet.

The next notebook should focus on **generator expressions** rather than repeating the basic generator introduction.